# DINOv3 ViT-L/16 — Unfolded-Attention CRP Walkthrough

End-to-end concept attribution + relevance maximisation on DINOv3
ViT-L/16 with the unfolded-attention refactor and the AlphaBeta
(0.5, 0.5) bilinear rule, using the six concept classes
(`HeadConcept`, `QConcept`, `KConcept`, `VConcept`,
`AttnOutputDimConcept`, `RegisterTokenConcept`).

## Setup — what to run before opening the notebook

**1. Install the environment.** From the repo root:

```bash
uv sync
```

All dependencies (torch, timm, lightning, jupyter, huggingface-hub,
etc.) are declared in `pyproject.toml` and installed into `.venv/`.
No optional extras needed for this notebook.

**2. Pick a (base, head, dataset).** Defaults below are
`vit_dinov3 + linear + funny_birds`; change them in section 2.
Available choices:

| | options |
|---|---|
| **base**    | `vit_base` *(timm vit_base_patch16_224)* &nbsp;·&nbsp; `vit_dinov3` *(timm vit_large_patch16_dinov3, default)* |
| **head**    | `linear` *(cls-token classifier, default)* &nbsp;·&nbsp; `attentive` *(learned-query attention pool, DINOv3-paper SOTA)* |
| **dataset** | `funny_birds` *(50 synthetic birds + GT part maps, ~1.5 GB)* &nbsp;·&nbsp; `dsprites` *(3 shapes, ~26 MB)* &nbsp;·&nbsp; `imagenet_val_hf` *(1000 classes, ~830 MB)* &nbsp;·&nbsp; `imagenette` *(10 classes, ~98 MB)* |

All datasets **auto-download** on first use — no manual setup.

**3. Cache features and train a probe head.** Two-step CLI
(installed by `uv sync` as `train-probe`) — extract features
once, train any number of head variants on top of the cache.
Substitute your `<base>`, `<head>`, `<dataset>` in:

```bash
# Step A — cache features (linear → cls, attentive → tokens).
# The dataset auto-downloads on first call; cache file lives at
# data/<base>_<dataset>_<kind>_feats.pt.
uv run train-probe cache <base> <dataset> --kind <cls|tokens>

# Step B — train the head on the cache. Lightning + ModelCheckpoint
# (best val_acc) + EarlyStopping. Output:
# data/<base>_<head>_probe_<dataset>.pt.
uv run train-probe train <base> <head> <dataset>
```

**Concrete example** — full setup for the default
`vit_dinov3 + linear + funny_birds`:

```bash
uv sync
uv run train-probe cache vit_dinov3 funny_birds --kind cls
uv run train-probe train vit_dinov3 linear funny_birds
```

Then open this notebook and run all cells. The probe-loading cell
(in section 2) raises `FileNotFoundError` with the exact two
commands to run if the checkpoint is missing — you'll never have
to dig through this header to remember the recipe.

**For the SOTA attentive head**:

```bash
uv run train-probe cache vit_dinov3 funny_birds --kind tokens  # ~20 GB
uv run train-probe train vit_dinov3 attentive funny_birds --num-heads 8
```

Then set `HEAD = 'attentive'` in section 2. The head trains with
vanilla PyTorch (no LRP behaviour leaks into training); the AttnLRP
rules — including the AlphaBeta bilinear at `α=β=0.5` — are
applied at attribution time by this notebook's composite, which
rebinds the head's `BilinearMatmul` / `SoftmaxAlongLastDim` /
`ScaleByConstant` forwards inside the composite context and
restores them on exit.

**4. (Optional) Hardware.** A single NVIDIA GPU with ≥24 GB VRAM
comfortably runs everything. CPU-only works for inspection but the
FV indexing cell (section 9) will be slow.

## Notebook structure

1. Setup (imports, repo paths)
2. Configuration — base × head × dataset, plus composite
3. Layer name reference — which submodules are hookable
4. Load dataset + pick a focal image
5. HeadConcept atlas at `HEAD_LAYER`
6. Q / K / V concept atlases at `Q_LAYER` / `K_LAYER` / `V_LAYER`
7. AttnOutputDimConcept top-K channel atlas
8. RegisterTokenConcept atlas (cls + 4 register tokens)
9. Build FV index for reference-sample retrieval
10. Reference samples per concept granularity
11. Conditional propagation cascade (HeadConcept across depth)
12. Notes & next steps

## 1. Setup

In [ ]:
%cd ../../..
%ls
from __future__ import annotations
import warnings
from pathlib import Path

# Walk up from the notebook's location to find the repo root —
# only used to point at <repo>/data/. No sys.path manipulation:
# `experiments` and `crp` are installable packages exposed by
# `uv sync` (project.scripts in pyproject.toml).
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').is_file():
    REPO_ROOT = REPO_ROOT.parent

import torch
import numpy as np
import matplotlib.pyplot as plt

from crp.attribution import CondAttribution
from crp.attention_concepts import (
    HeadConcept, QConcept, KConcept, VConcept,
    AttnOutputDimConcept, RegisterTokenConcept,
)
from crp.transformer_patches import AttnLRPCombinedComposite
from crp.visualization import FeatureVisualization
from crp.image import imgify

from experiments.datasets import load as load_dataset
from experiments.models import BASES, HEADS, build_probe  # same registry
                                              # the CLI uses — rebuilding
                                              # a trained probe is just
                                              # build_probe(...) + load_state_dict().
from experiments.viz_unfolded import (
    to_display, panel,
    plot_concept_atlas, plot_cascade,
    enumerate_ids, label_id, row_label,
    attribute_at_concept, per_concept_scores,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')
print(f'available bases: {list(BASES)}')
print(f'available heads: {list(HEADS)}')

## 2. Configuration — base × head × dataset, plus composite

**Composite** — the recipe validated in `RESEARCH_NOTES.md` Entry 6:

* `matmul_factor_2=True` — bilinear matmul rule (AttnLRP Prop 3.3)
* `alpha=0.5, beta=0.5` — AlphaBeta variant of the bilinear rule
  (Bach 2015 generalised to bilinear — best magnitude control,
  ~19 OOM tighter than the standard `2y+ε` rule on DINOv3 ViT-L)
* `layerscale_uniform=True` — uniform-rule LayerScale γ allocation.
  LayerScale (Touvron et al. CaiT 2021) multiplies each branch by
  γ ≈ 1e-4. Bare backward gives `grad_branch = grad_y · γ` —
  multiplying by γ deflates relevance massively per layer (over
  24 blocks × 2 LayerScales/block this compounds to numerical
  death). `layerscale_uniform=True` wraps `γ * branch` in
  `divide_gradient(., 2)` — the AttnLRP uniform rule (Eq. 7):
  γ (a leaf parameter) absorbs half the relevance, branch's half
  (= R/2) propagates back through the chain. Replaces `×γ`
  dampening with constant `×½` per LayerScale, keeping
  magnitudes alive through deep stacks.
* `residual_lrp='ratio'` — Otsuki ratio split on residual additions
* `use_unfolded_attention=True` — substitute EvaAttention with
  EvaAttentionUnfolded (required for concept conditioning)

**Base × head × dataset** — pick one of each. The model is
(re)built from the same registry the training CLI uses, so swapping
to a different head (e.g. `attentive`) is just changing one variable.

In [ ]:
# === MODEL × DATASET CHOICE ===
# Comment / uncomment ONE option block. Both substitute attention to
# the unfolded form (Eva ↔ EvaAttentionUnfolded, timm ↔
# TimmAttentionUnfolded), so concept atlases work on either backbone —
# only the per-concept Q/K hook names differ (rope_q on Eva, q_relevance_inspection_point on
# stock timm).

# ── Option 1 (default): DINOv3 ViT-L + frozen linear probe ──────────
#BASE = 'vit_dinov3'
#HEAD = 'linear'      # 'linear', 'attentive', or 'block'
#DATASET = 'funny_birds'
#ATTN_Q_LEAF = 'rope_q'    # DINOv3 has RoPE → Q tap is post-RoPE
#ATTN_K_LEAF = 'rope_k'

# ── Option 2: FunnyBirds-pretrained vit_base (visinf weights) ──────
# Setup: `uv run python experiments/scripts/setup_funnybirds_vit_base.py`
# Standard timm Attention has no RoPE → Q/K concepts hook the post-norm
# Identity placeholders (q_relevance_inspection_point / k_relevance_inspection_point).
BASE = 'vit_base'
HEAD = 'linear'
DATASET = 'funny_birds'
ATTN_Q_LEAF = 'q_relevance_inspection_point'
ATTN_K_LEAF = 'k_relevance_inspection_point'

DATA_ROOT = REPO_ROOT / 'data'
PROBE_PATH = DATA_ROOT / f'{BASE}_{HEAD}_probe_{DATASET}.pt'

# FV indexing: by default we index the WHOLE dataset (set in section 9
# once `dataset` is loaded). For fast dev iteration, override there
# with e.g. `fv.run(composite, 0, 500, …)` to index only the first
# 500 samples.

# Composite — the validated AlphaBeta recipe. Both substitution
# canonizers (Eva + Timm) are always registered; each one's
# isinstance filter no-ops on the other's target class.
COMPOSITE_KWARGS = dict(
    alpha=0.5, beta=0.5,
    layerscale_uniform=True,   # no-op on vit_base (no LayerScale); active on DINOv3
    residual_lrp='ratio',
)

# Per-concept layer paths (HEAD_LAYER, Q_LAYER, …, CASCADE_LAYER_NAMES)
# are set in the build-model cell below — they need `len(model.blocks)`.

# How many concepts to visualise per atlas.
TOP_K_CONCEPTS = 6

# How many reference samples per concept.
N_REFS = 4

# Random seed for sample selection.
RANDOM_SEED = 0

# Sanity-eval subset size (section 2 end). Tighter estimate with
# larger N at the cost of more time.
EVAL_SUBSET_SIZE = 200

print(f'base    : {BASE}')
print(f'head    : {HEAD}')
print(f'dataset : {DATASET}')
print(f'probe   : {PROBE_PATH}')

In [ ]:
# Load the probe checkpoint and rebuild the full model with the same
# `build_probe` registry the training CLI uses. If the probe is
# missing, the cell prints the exact two commands to make it.
if not PROBE_PATH.is_file():
    head_kind = HEADS[HEAD].input_kind
    raise FileNotFoundError(
        f'\nProbe checkpoint not found at {PROBE_PATH}.\n\n'
        f'Step 1 — cache features (one-shot, reusable across heads):\n'
        f'  uv run train-probe cache {BASE} {DATASET} --kind {head_kind}\n\n'
        f'Step 2 — train the head:\n'
        f'  uv run train-probe train {BASE} {HEAD} {DATASET}\n'
    )
ckpt = torch.load(PROBE_PATH, map_location=DEVICE, weights_only=False)
print(f'probe trained on : {ckpt["dataset"]}'
      f' ({ckpt["num_classes"]} classes)')
_va, _va5 = ckpt.get('val_acc'), ckpt.get('val_acc5')
print(f'val_acc          : {_va:.4f}' if _va is not None else 'val_acc          : (unreported — see sanity-eval below)')
print(f'val_acc5         : {_va5:.4f}' if _va5 is not None else 'val_acc5         : (unreported)')

In [ ]:
from timm.data import resolve_data_config, create_transform
import torch.nn.functional as F
from torchvision.transforms import functional as TF

# Build the full Probe (frozen base + trainable head) via the same
# registry used by the CLI. Backbone is loaded fresh from timm via
# build_probe → Base.__init__; trained head weights come from the ckpt.
model = build_probe(
    base=ckpt['base'], head=ckpt['head'],
    num_classes=ckpt['num_classes'],
    head_kwargs=ckpt.get('head_kwargs', {}),
).eval().to(DEVICE)
model.head.load_state_dict(ckpt['head_state_dict'])
if 'backbone_state_dict' in ckpt:
    # Fine-tuned or externally-trained backbone (e.g. setup_funnybirds_vit_base.py
    # or train_probe finetune_cmd output) overrides whatever the Base class
    # loaded at construction.
    model.backbone.load_state_dict(ckpt['backbone_state_dict'])
    print('  loaded backbone weights from ckpt (pretrained / finetuned)')
for p in model.parameters():
    p.requires_grad_(False)

# Build the eval transform AND the per-batch normalize callable. The
# dataset transform produces unnormalized [0, 1] tensors (display-
# ready, uniform across DataLoader / FeatureVisualization / Lightning
# / raw forward); normalize is applied at the forward boundary so the
# model sees its expected input distribution. This split lets external
# pretrained checkpoints (e.g. visinf vit_base, trained without
# normalize) coexist with timm-default models in the same notebook
# without any denormalize indirection for display.
TRANSFORM_SPEC = ckpt.get('transform_spec', 'timm_default')
if 'transform_spec' not in ckpt:
    print('⚠ ckpt has no `transform_spec` field — falling back to timm_default.')
    if ckpt.get('base') == 'vit_base' and ckpt.get('dataset') == 'funny_birds':
        print('  Your payload looks like the visinf vit_base FunnyBirds checkpoint.')
        print('  Re-run the setup script to refresh the payload with transform_spec:')
        print('    uv run python experiments/scripts/setup_funnybirds_vit_base.py --force')
        print('  Then re-run this cell.')

if TRANSFORM_SPEC == 'timm_default':
    # Resize + crop + ToTensor from timm cfg; mean/std overridden to
    # identity so the timm transform's normalize step is a no-op.
    # The actual normalize stats (read from the registered cfg) ship
    # to the model in a separate callable.
    cfg = resolve_data_config({}, model=model.backbone)
    _resize_cfg = {**cfg, 'mean': (0.0, 0.0, 0.0), 'std': (1.0, 1.0, 1.0)}
    transform = create_transform(**_resize_cfg, is_training=False)
    _mean = torch.tensor(cfg['mean']).view(1, -1, 1, 1).to(DEVICE)
    _std  = torch.tensor(cfg['std']).view(1, -1, 1, 1).to(DEVICE)
    def normalize(x):
        return (x - _mean) / _std
    print(f'transform  : timm_default (resize-only; normalize mean={cfg["mean"]}, std={cfg["std"]})')

elif TRANSFORM_SPEC == 'visinf_funnybirds_vit_base':
    # visinf/funnybirds-framework trains its ViT with `transforms=None`
    # in train.py (only `torchvision.transforms.ToTensor` → [0,1])
    # — NO ImageNet/JFT normalize. Their model wrapper resizes 256→224
    # via `F.interpolate(x, (224, 224))` (defaults to nearest). We use
    # bilinear here for cleaner image resampling — both give 0.98–0.99
    # top-1 on the FunnyBirds test set (within sampling noise of the
    # checkpoint's own `best_acc1=98.0`); the (0.5,0.5,0.5)-normalize +
    # crop_pct=0.9 timm default drops it to ~0.85. The model expects
    # raw [0, 1] inputs → `normalize` is the identity.
    def transform(pil):
        t = TF.to_tensor(pil)[:3]  # drops alpha, keeps [0,1] floats
        t = F.interpolate(t.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False).squeeze(0)
        return t
    def normalize(x):
        return x
    print('transform  : visinf_funnybirds_vit_base (resize-only; no-normalize)')

else:
    raise ValueError(f'unknown transform_spec {TRANSFORM_SPEC!r} — register a branch in the build-model cell')

_npt = int(getattr(model, 'num_prefix_tokens', 1))
print(f'base       : {ckpt["base"]}')
print(f'head       : {ckpt["head"]} kwargs={ckpt.get("head_kwargs", {})}')
print(f'embed_dim  : {model.embed_dim}')
print(f'num_blocks : {len(model.blocks)}')
print(f'num_heads  : {model.blocks[0].attn.num_heads}')
print(f'head_dim   : {model.blocks[0].attn.head_dim}')
if _npt > 1:
    print(f'num_prefix : {_npt} (1 cls + {_npt - 1} register)')
else:
    print(f'num_prefix : {_npt} (cls only)')

# Per-concept layer paths — each is a string from `model.named_modules()`.
# Defaults point at the LAST attention block (closest to the
# classification head, most likely carrying object-semantic concepts).
# ATTN_Q_LEAF / ATTN_K_LEAF come from the selector cell — they differ
# between Eva (rope_q/rope_k) and standard timm (q_relevance_inspection_point/k_relevance_inspection_point).
# Replace any string with another `.named_modules()` path whose
# tensor shape matches the concept's contract — see section 3's
# discovery cell for the available paths.
_LAST = len(model.blocks) - 1
HEAD_LAYER = f'backbone.blocks.{_LAST}.attn.context'
Q_LAYER    = f'backbone.blocks.{_LAST}.attn.{ATTN_Q_LEAF}'
K_LAYER    = f'backbone.blocks.{_LAST}.attn.{ATTN_K_LEAF}'
V_LAYER    = f'backbone.blocks.{_LAST}.attn.v_relevance_inspection_point'
OUT_LAYER  = f'backbone.blocks.{_LAST}.attn.proj_drop'
# RegisterTokenConcept needs a NON-LAST block: the classification head
# reads only the cls token, so register-token outputs at the last block
# have a strict mathematical zero of LRP relevance (no path to the
# logit). Earlier blocks DO carry signal because register-token outputs
# feed into the cls token at deeper blocks via attention. Default to
# the second-to-last block — rich enough to show structure, still near
# the head so concepts are object-semantic rather than texture.
REG_LAYER  = f'backbone.blocks.{max(_LAST - 1, 0)}.attn.proj_drop'

# Cascade: deep → shallow attention.context paths. Arithmetic on _LAST
# so it works for any depth (vit_base 12 blocks, vit_dinov3 24 blocks).
# Replace with any list of `.named_modules()` paths whose tensor shape
# matches the cascade concept's contract (HeadConcept → 'attn.context').
CASCADE_LAYER_NAMES = [
    f'backbone.blocks.{i}.attn.context'
    for i in [_LAST, _LAST * 3 // 4, _LAST // 2, _LAST // 4, 0]
]

In [ ]:
# Composite: the validated AlphaBeta recipe. Both substitution
# canonizers (Eva + Timm) are registered unconditionally; each fires
# only on its own attention class.
composite = AttnLRPCombinedComposite(**COMPOSITE_KWARGS)
attribution = CondAttribution(model)
print(f'composite: {composite}')

## 3. Layer name reference (which submodules are hookable)

Each unfolded EvaAttention block exposes named submodules you can
target by string. The discovery cell below prints the actual paths
as `model.named_modules()` reports them — these are the strings to
use as `condition` keys and `record_layer` arguments.

Concept ↔ submodule mapping (see the concept-class docstrings for
the rationale of each choice):

| Concept | Submodule |
|---|---|
| `HeadConcept`           | `attn.context` |
| `QConcept`              | `attn.rope_q` |
| `KConcept`              | `attn.rope_k` |
| `VConcept`              | `attn.v_relevance_inspection_point` |
| `AttnOutputDimConcept`  | `attn.proj_drop` (spatial tokens) |
| `RegisterTokenConcept`  | `attn.proj_drop` (prefix tokens) |

In [ ]:
# Discover the unfolded attention's named submodules — these are the
# leaf strings users append onto the attention block path to address
# a specific tensor with `record_layer=[...]` or in a condition dict.
with composite.context(model) as modified:
    attn_path = 'backbone.blocks.0.attn'
    attn0 = modified.get_submodule(attn_path)
    print(f'Hookable submodules under {attn_path!r}:')
    for name, _ in attn0.named_children():
        print(f'  {attn_path}.{name}')

print()
print(f'Available block indices: 0 .. {len(model.blocks) - 1}')
print('Per-concept defaults (set in the build-model cell):')
for name, path in [('HEAD_LAYER', HEAD_LAYER), ('Q_LAYER', Q_LAYER),
                   ('K_LAYER', K_LAYER), ('V_LAYER', V_LAYER),
                   ('OUT_LAYER', OUT_LAYER), ('REG_LAYER', REG_LAYER)]:
    print(f'  {name:10s} = {path}')
print(f'  CASCADE_LAYER_NAMES = {CASCADE_LAYER_NAMES}')

### Single-block computation graph

Visualize one attention block's unfolded internals as a directed
graph. The composite context substitutes the stock attention with
`EvaAttentionUnfolded` / `TimmAttentionUnfolded`, exposing every
atomic op (`qkv`, `q_norm`, `q_relevance_inspection_point`/`rope_q`, `scale_q`, `qk_scores`,
`softmax`, `context`, `proj`, …) as a named child module. The named
submodules in the rendered graph are exactly the strings you'd pass
to `record_layer=[...]` or use in a condition dict.

Renders via [`torchview`](https://github.com/mert-kurttutan/torchview),
which calls system Graphviz to produce an SVG. If `graphviz` isn't
installed, falls back to printing the named-submodule tree.
Install graphviz with `apt install graphviz` / `brew install graphviz`.

In [ ]:
BLOCK_TO_VIZ = 0  # block index to render — same graph for all blocks

try:
    from torchview import draw_graph
    _can_torchview = True
except ImportError:
    print('torchview not installed; run `uv pip install torchview` and re-run.')
    _can_torchview = False

if _can_torchview:
    # Substitute the attention with its unfolded variant via the composite
    # context — that's the graph we actually run attribution against.
    with composite.context(model) as modified:
        block = modified.backbone.blocks[BLOCK_TO_VIZ]
        # Probe the right token-sequence shape from the model.
        _N = _npt + int(getattr(model, 'num_patches', 196))
        _x = torch.randn(1, _N, model.embed_dim, device=DEVICE)
        try:
            g = draw_graph(
                block, input_data=_x,
                depth=3, expand_nested=True,
                graph_name=f'block_{BLOCK_TO_VIZ}',
                hide_module_functions=False,
            )
            # Render to in-memory SVG. Requires system `graphviz` (the
            # `dot` binary). On a missing binary the call raises;
            # we catch and fall back to a structural print below.
            from IPython.display import display, SVG
            svg_str = g.visual_graph.pipe(format='svg').decode('utf-8')
            display(SVG(svg_str))
        except Exception as e:
            print(f'graphviz render failed ({type(e).__name__}: {e!s}).')
            print(f'Falling back to named-submodule tree of block {BLOCK_TO_VIZ}:')
            print('  ' + '\n  '.join(
                f'{name}: {type(m).__name__}'
                for name, m in block.named_modules() if name
            ))

#### Canonized-block sanity check

The torchview render can look like it has dead ends — that's a
**graph-layout artifact**, not a real disconnection. This cell is
the trustworthy ground truth:

1. **Forward parity** — the substituted block (unfolded attention)
   must produce the same output tensor as the stock block on the
   same input (within fp32 noise).
2. **No dead ends** — every named submodule's output must have a
   nonzero gradient w.r.t. the block output. If anything has zero
   or `None` gradient, it's dead and a real bug.

In [ ]:
import torch as _torch_chk

with _torch_chk.no_grad():
    _stock_block = model.backbone.blocks[BLOCK_TO_VIZ]
    _x_stock = _torch_chk.randn(1, _N, model.embed_dim, device=DEVICE)
    _y_stock = _stock_block(_x_stock)

with composite.context(model) as _modified_for_check:
    _sub_block = _modified_for_check.backbone.blocks[BLOCK_TO_VIZ]

    # 1. Forward parity
    with _torch_chk.no_grad():
        _y_sub = _sub_block(_x_stock)
    _max_diff = (_y_sub - _y_stock).abs().max().item()
    _stock_sum, _sub_sum = _y_stock.sum().item(), _y_sub.sum().item()
    _ok_fwd = _max_diff < 1e-2 * max(abs(_stock_sum), 1.0)
    print(f'Forward parity: stock sum={_stock_sum:.4f}, substituted sum={_sub_sum:.4f}')
    print(f'  max element-wise diff = {_max_diff:.3e}  → {"OK" if _ok_fwd else "⚠ MISMATCH"}')

    # 2. Dead-end check — register forward hooks on every named submodule
    _captured = {}
    _handles = []
    for _name, _m in _sub_block.named_modules():
        if not _name:
            continue
        def _hook(_mod, _inp, _out, __n=_name):
            if isinstance(_out, _torch_chk.Tensor) and _out.requires_grad:
                _out.retain_grad()
                _captured[__n] = _out
        _handles.append(_m.register_forward_hook(_hook))
    try:
        _x_grad = _torch_chk.randn(1, _N, model.embed_dim, device=DEVICE, requires_grad=True)
        _y_grad = _sub_block(_x_grad)
        _y_grad.sum().backward()
    finally:
        for _h in _handles:
            _h.remove()
    _dead = [
        n for n, t in _captured.items()
        if t.grad is None or t.grad.abs().sum().item() == 0
    ]
    print(f'\nDead-end check on {len(_captured)} hookable submodules:')
    if _dead:
        print(f'  ⚠ {len(_dead)} dead end(s) — these submodules do NOT contribute to the block output:')
        for n in _dead:
            print(f'    {n}')
    else:
        print(f'  ✓ all {len(_captured)} submodules contribute to the block output')

## 4. Load dataset + pick a focal image

The chosen `DATASET` is loaded via the unified `load(name, ...)`
dispatcher in `experiments/datasets/`. Each dataset module handles
its own download/extract/setup automatically. The focal image is
the first correctly-classified sample under the trained probe;
everything below attributes against this image's predicted class.

In [ ]:
# Build the dataset variants used downstream. For FunnyBirds we expose
# three named handles so it's easy to switch what the FV cache (sec 9)
# and the cascade (sec 11) operate on:
#
#   dataset_test          : held-out test split (500 imgs, 0% ablations)
#   dataset_train_clean   : train filtered to intact birds (~29k imgs)
#   dataset_train_ablated : full train, includes part-ablations (~50k,
#                           ~41% have one or more body parts replaced
#                           with 'placeholder' renders).
#
# Downstream cells read `dataset` — change the assignment below to
# switch the FV indexing pool. The sanity-eval cell uses
# `dataset_eval` independently (always the clean test split for
# FunnyBirds — matches the advertised accuracy figure).
if DATASET == 'funny_birds':
    dataset_test          = load_dataset('funny_birds', transform=transform, split='test')
    dataset_train_clean   = load_dataset('funny_birds', transform=transform, split='train', clean_only=True)
    dataset_train_ablated = load_dataset('funny_birds', transform=transform, split='train', clean_only=False)
    dataset_eval = dataset_test
    # Default: ablated train — gives the FV cache access to part-
    # ablation samples, so the resulting concept references reveal
    # whether any head/channel fires specifically on missing-part
    # renders. Switch to `dataset_train_clean` for a baseline FV that
    # mirrors the test distribution.
    dataset = dataset_train_ablated
    print(f'  dataset_test          : {len(dataset_test):>6d} imgs')
    print(f'  dataset_train_clean   : {len(dataset_train_clean):>6d} imgs')
    print(f'  dataset_train_ablated : {len(dataset_train_ablated):>6d} imgs')
else:
    _load_kwargs = {
        'dsprites':        dict(target='shape'),
        'imagenette':      dict(split='val'),
        'imagenet_val_hf': dict(),
    }[DATASET]
    dataset_eval = dataset = load_dataset(DATASET, transform=transform, **_load_kwargs)
print(f'downstream `dataset`      : {len(dataset):>6d} imgs')
print(f'sanity-eval `dataset_eval`: {len(dataset_eval):>6d} imgs')

### Sanity-eval: does the loaded model classify this dataset?

Run the model on a small slice of `dataset_eval` (the held-out test
split for FunnyBirds, which by construction has 0% ablations). A
near-chance number here means the checkpoint didn't load correctly
(wrong head shape, missing backbone weights, base/dataset mismatch, or
a transform mismatch between the checkpoint's training pipeline and
our eval pipeline). Bump `EVAL_SUBSET_SIZE` (in section 2) for tighter
estimates.

In [ ]:
from torch.utils.data import DataLoader, Subset

n_eval = min(EVAL_SUBSET_SIZE, len(dataset_eval))
_loader = DataLoader(
    Subset(dataset_eval, list(range(n_eval))),
    batch_size=32, shuffle=False, num_workers=2,
)
_correct = _total = 0
with torch.no_grad():
    for _x, _y in _loader:
        # Dataset yields unnormalized tensors; normalize at the forward
        # boundary.
        _logits = model(normalize(_x.to(DEVICE)))
        _correct += (_logits.argmax(-1).cpu() == _y).sum().item()
        _total   += _y.numel()
print(f'Sanity-eval top-1 over {_total} samples of dataset_eval: {_correct/_total:.4f}')
if 'val_acc' in ckpt and ckpt['val_acc'] is not None:
    print(f'  (advertised val_acc in payload: {ckpt["val_acc"]:.4f})')

### Pick the focal image

In [ ]:
# `focal_image` holds the UNNORMALIZED [0, 1] tensor — display-ready,
# safe to pass to viz_unfolded funcs (which `preprocess_fn=normalize`).
# Model forwards inside this cell normalize at the boundary.
rng = np.random.default_rng(RANDOM_SEED)
stride = max(1, len(dataset) // 30)
focal_image = focal_class = focal_index = None
for i in range(0, len(dataset), stride):
    x_, y_ = dataset[i]
    x_dev = x_.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = model(normalize(x_dev)).argmax(-1).item()
    if pred == int(y_):
        focal_image = x_dev.detach().requires_grad_(True)
        focal_class = pred
        focal_index = i
        break
if focal_image is None:
    raise RuntimeError(
        f'No correctly-classified sample found in first {len(dataset)//stride} '
        f'strided samples — probe accuracy may be too low. Re-train it with '
        f'more epochs or check dataset compatibility.'
    )

print(f'focal image: dataset index {focal_index}, class {focal_class}')
fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.imshow(to_display(focal_image)); ax.axis('off')
ax.set_title(f'class {focal_class}')
plt.show()

### Plain attribution (no concept conditioning) for reference

In [ ]:
# Attribution runs on the NORMALIZED tensor — that's what the model
# expects internally. The display tensor stays unnormalized.
x_run = normalize(focal_image).detach().clone().requires_grad_(True)
res = attribution(x_run, [{'y': [focal_class]}], composite)
hm = res.heatmap[0]
if hm.dim() == 3 and hm.shape[0] == 3:
    hm = hm.sum(dim=0)
fig, ax = plt.subplots(1, 1, figsize=(4, 2))
panel(ax, to_display(focal_image), hm.detach().cpu().numpy())
ax.set_title(f'plain attribution toward class {focal_class}', fontsize=9)
plt.show()

## 5. HeadConcept atlas at `HEAD_LAYER`

One panel per attention head at the chosen layer. Each panel shows
the input-space heatmap obtained by conditioning the backward on
that single head's `attn @ V` output. `score` in the title is the
per-head relevance summed over spatial tokens (excludes register
tokens by design).

In [ ]:
# Concept just stores a reference to the model and reads dims (num_heads,
# head_dim, num_prefix_tokens) from the attention parent at every call.
# Works whether the model is bare or in a composite context.
head_concept = HeadConcept(model)
fig = plot_concept_atlas(
    focal_image, model, attribution, composite,
    concept=head_concept, layer_name=HEAD_LAYER,
    target_class=focal_class, top_k=TOP_K_CONCEPTS,
    preprocess_fn=normalize,
)
plt.show()

## 6. Q / K / V concept atlases at `Q_LAYER` / `K_LAYER` / `V_LAYER`

Same per-head granularity, but conditioned at the Q (post q_norm + RoPE),
K (post k_norm + RoPE), V (post per-head reshape) inputs to the
attention bilinears. These three rows show what each head's
**query / key / value** subspace contributes to the prediction.

In [ ]:
for label, concept, layer in [
    ('Q', QConcept(model), Q_LAYER),
    ('K', KConcept(model), K_LAYER),
    ('V', VConcept(model), V_LAYER),
]:
    fig = plot_concept_atlas(
        focal_image, model, attribution, composite,
        concept=concept, layer_name=layer,
        target_class=focal_class, top_k=TOP_K_CONCEPTS,
        title_prefix=label,
        preprocess_fn=normalize,
    )
    plt.show()

## 7. AttnOutputDimConcept top-K channel atlas

Conditioning at the **post-projection** residual contribution
(i.e. what the attention block writes into the residual stream).
Per-channel, spatial-aggregated. With `embed_dim = 1024` we
show only the top-K most relevant channels.

In [ ]:
out_concept = AttnOutputDimConcept(model)
fig = plot_concept_atlas(
    focal_image, model, attribution, composite,
    concept=out_concept, layer_name=OUT_LAYER,
    target_class=focal_class, top_k=TOP_K_CONCEPTS,
    preprocess_fn=normalize,
)
plt.show()

## 8. RegisterTokenConcept atlas — cls + register tokens

DINOv3 prepends 5 prefix tokens (1 cls + 4 register). Each
carries global non-spatial signal — register tokens absorb
high-norm artifacts (Darcet et al. ICLR 2024,
arXiv:2309.16588). Per-token conditioning shows what each
prefix token contributes via the `proj_drop` residual stream.

**Why the heatmaps are spatial.** Prefix tokens have no spatial
location of their own, but they are *populated* by attending to
spatial patch positions inside each attention block
(`token_value[k] = Σᵢ attn[token, i] · V[i, k]`). LRP-backward
from a masked prefix-token output routes through `context = attn @ V`,
spreading relevance to all spatial V positions weighted by that
token's attention row. The resulting input-space heatmap answers:
*"which input pixels populated this prefix token at this layer?"*

**Why we don't use the LAST block here.** The classification head
reads only the cls token (position 0), so register-token outputs
at the last block have a strict mathematical zero of relevance
(no downstream consumer). `REG_LAYER` defaults to the **second-
to-last** block — register tokens at that depth still influence
the head (via attention from cls at the next block) and so carry
non-trivial spatial heatmaps.

In [ ]:
reg_concept = RegisterTokenConcept(model)
fig = plot_concept_atlas(
    focal_image, model, attribution, composite,
    concept=reg_concept, layer_name=REG_LAYER,
    target_class=focal_class,  # show ALL prefix tokens (5 cells)
    preprocess_fn=normalize,
)
plt.show()

## 9. Build FV index for reference-sample retrieval

Computes per-concept relevance over the whole loaded dataset (default:
`fv_end = len(dataset)`) and caches the per-concept top-N images for
fast retrieval. We index three granularities at the focal layer:

* `HeadConcept` at `HEAD_LAYER` — "images that maximally activate head k"
* `AttnOutputDimConcept` at `OUT_LAYER` — "images that activate channel c"
* `RegisterTokenConcept` at `REG_LAYER` — "images that maximally drive a prefix token"

For a fast smoke-test override `fv_end` with a small integer (e.g.
500). The cache is persisted to `data/fv_cache_dinov3_unfolded/<tag>`.
**Re-running the cell is safe**: if `<tag>/RelMax_*/*_data.npy` already
exists, the indexing step is skipped and the `FeatureVisualization`
object is just rebound to the on-disk index. To force a rebuild, delete
the matching `<tag>` folder (or change `FV_CACHE_DIR`).

In [ ]:
FV_CACHE_DIR = REPO_ROOT / 'data' / 'fv_cache_dinov3_unfolded'
FV_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Index over the ENTIRE dataset so each concept's top-N reference
# samples are picked from the largest possible pool. For fast dev,
# replace `fv_end = len(dataset)` with a small integer.
fv_end = len(dataset)

# FV-index targets: tuples of (cache-tag, concept-instance, list-of-
# target-layers). Each entry produces ONE FeatureVisualization whose
# layer_map covers every listed layer. FV runs one backward pass per
# image and books-keeps activations per layer in that pass — adding
# more layers to a tag costs only the bookkeeping, not extra compute.
#
# Quick defaults (uncommented): head/attnoutdim/register at the focal
# (last) block; HeadConcept ALSO at every cascade depth so section 11
# can display references. Uncomment any of the alternatives below for
# Q/K/V or for full-depth indexing — only the missing tags get
# re-indexed (existing on-disk indexes are reused).

# Block-path prefixes for the cascade depths and the full model. Use
# these in the alternatives below if you want Q/K/V references at
# the same depths the cascade walks, or at every block.
_BLOCKS         = [f'backbone.blocks.{i}' for i in range(len(model.blocks))]
_CASCADE_BLOCKS = [l.rsplit('.attn.', 1)[0] for l in CASCADE_LAYER_NAMES]
_HEAD_CASCADE   = list(dict.fromkeys([HEAD_LAYER, *CASCADE_LAYER_NAMES]))

fv_specs = [
    # ── quick defaults ──────────────────────────────────────────────
    ('head',       HeadConcept(model),          _HEAD_CASCADE),
    ('attnoutdim', AttnOutputDimConcept(model), [OUT_LAYER]),
    ('register',   RegisterTokenConcept(model), [REG_LAYER]),
    # ── extra: Q/K/V at the focal block only ────────────────────────
    # ('q', QConcept(model), [Q_LAYER]),
    # ('k', KConcept(model), [K_LAYER]),
    # ('v', VConcept(model), [V_LAYER]),
    # ── extra: Q/K/V references at every cascade depth ──────────────
    # Useful if you swap the cascade concept (section 11) for Q/K/V.
    # ('q_cascade', QConcept(model), [f'{b}.attn.{ATTN_Q_LEAF}' for b in _CASCADE_BLOCKS]),
    # ('k_cascade', KConcept(model), [f'{b}.attn.{ATTN_K_LEAF}' for b in _CASCADE_BLOCKS]),
    # ('v_cascade', VConcept(model), [f'{b}.attn.v_relevance_inspection_point'         for b in _CASCADE_BLOCKS]),
    # ── extra: every concept at EVERY block (full depth scan) ───────
    # Heavy — multiplies index size by `len(model.blocks)`.
    # ('head_all',       HeadConcept(model),          [f'{b}.attn.context'       for b in _BLOCKS]),
    # ('attnoutdim_all', AttnOutputDimConcept(model), [f'{b}.attn.proj_drop'     for b in _BLOCKS]),
    # ('register_all',   RegisterTokenConcept(model), [f'{b}.attn.proj_drop'     for b in _BLOCKS]),
    # ('q_all',          QConcept(model),             [f'{b}.attn.{ATTN_Q_LEAF}' for b in _BLOCKS]),
    # ('k_all',          KConcept(model),             [f'{b}.attn.{ATTN_K_LEAF}' for b in _BLOCKS]),
    # ('v_all',          VConcept(model),             [f'{b}.attn.v_relevance_inspection_point'          for b in _BLOCKS]),
]

fv_results = {}
for tag, concept, layers in fv_specs:
    cache_path = FV_CACHE_DIR / tag
    fv = FeatureVisualization(
        attribution=attribution, dataset=dataset,
        layer_map={layer: concept for layer in layers},
        preprocess_fn=normalize,  # applied to each batch before forward
        path=str(cache_path), device=DEVICE,
    )
    # Skip only if EVERY listed layer already has a *_data.npy on disk.
    # Adding a new layer to an existing tag re-runs the indexing; the
    # cost is one full pass over the dataset (forward+backward), shared
    # across all layers in the tag's layer_map.
    has_index = cache_path.is_dir() and all(
        list(cache_path.rglob(f'{layer}_data.npy')) for layer in layers
    )
    if has_index:
        print(f'skipping {tag}: all {len(layers)} layer(s) already indexed at {cache_path}')
    else:
        print(f'indexing {tag} on {len(layers)} layer(s) over {fv_end} images …')
        for l in layers[:3]:
            print(f'  • {l}')
        if len(layers) > 3:
            print(f'  • … and {len(layers) - 3} more')
        fv.run(composite, 0, fv_end, batch_size=4, checkpoint=999)
    fv_results[tag] = (fv, concept, layers)
print('FV indexing complete')

## 10. Reference samples per concept granularity

For each indexed concept type, fetch the top-N images that maximally
activate the top-K concept ids at the focal layer. The grid below
is rows × N: each row is one concept id, each column one of the top
reference images.

In [ ]:
def show_references(fv, concept, layer, top_concept_ids, n_refs=N_REFS):
    # `get_max_reference` uses FV's default plot_fn (`vis_img_heatmap`),
    # which returns a tuple `(img_list, heat_list)` of PIL images per
    # concept id. We only display the input-space images here; pass
    # the heatmaps to imshow alongside if you want the side-by-side view.
    n = len(top_concept_ids)
    fig, axes = plt.subplots(n, n_refs, figsize=(1.2 * n_refs, 1.2 * n))
    if n == 1: axes = axes.reshape(1, -1)
    ref = fv.get_max_reference(
        top_concept_ids, layer, mode='relevance',
        r_range=(0, n_refs), composite=composite, rf=False,
    )
    for row, cid in enumerate(top_concept_ids):
        img_list, _heat_list = ref[cid]
        for ax, im in zip(axes[row], img_list[:n_refs]):
            ax.imshow(im); ax.axis('off')
        # Two-line row label: trimmed layer path on line 1, concept id
        # on line 2. e.g. 'blocks.22.attn.context' / 'H4' for HeadConcept.
        axes[row, 0].set_ylabel(
            row_label(concept, cid, layer),
            fontsize=7, rotation=0, ha='right', va='center',
            family='monospace', labelpad=8,
        )
    # Reserve left margin for the longer row labels.
    fig.subplots_adjust(left=0.22)
    fig.suptitle(f'{type(concept).__name__} top-{n_refs} refs', fontsize=9)
    plt.tight_layout()
    plt.show()

# For each indexed concept, pick the top-K most relevant ids on the focal
# image and show the reference samples. The first layer in each fv_spec's
# layers list is treated as the focal layer for this display.
for tag, (fv, concept, layers) in fv_results.items():
    layer = layers[0]
    scores = per_concept_scores(
        attribution, composite, focal_image, layer, concept, focal_class,
        preprocess_fn=normalize,
    )
    top_ids_idx = torch.argsort(scores.abs(), descending=True)[:TOP_K_CONCEPTS].cpu().tolist()
    all_ids = enumerate_ids(concept, layer)
    top_ids = [all_ids[i] for i in top_ids_idx]
    print(f'{tag}: top-{TOP_K_CONCEPTS} ids = {top_ids}')
    show_references(fv, concept, layer, top_ids, n_refs=N_REFS)

## 11. Conditional propagation cascade

Walk attention layers from deep to shallow. At each layer, condition
on the top-K most relevant concepts at every deeper layer already
visited (cumulative conditioning), then pick this layer's top-K.
Renders one row per layer; columns are the kept concepts. Reading
top-to-bottom traces how the model's attention concept selection
narrows as we descend toward the input.

Configurable: `CASCADE_LAYER_NAMES` (set in the build-model cell). Default uses
HeadConcept; swap in QConcept / VConcept etc. by changing the
concept argument.

**Reference-sample row.** After the cascade picks heads per layer, we
look up each (layer, head) pair in the prebuilt FV index (section 9)
and show the top-N images that maximally activate that head at that
depth. This requires every cascade layer to be indexed under the
`head` tag — the default `fv_specs` already does this.

In [ ]:
cascade_concept = HeadConcept(model)
fig, selected = plot_cascade(
    focal_image, model, attribution, composite,
    concept=cascade_concept,
    layer_names=CASCADE_LAYER_NAMES,
    target_class=focal_class,
    top_k=4,
    preprocess_fn=normalize,
)
plt.show()
print('selected concepts per layer (deep → shallow):')
for layer in CASCADE_LAYER_NAMES:
    print(f'  {layer}  →  heads {selected.get(layer)}')

### Reference samples for the cascade-selected heads

One row per cascade layer (deep → shallow), one column block per
selected head, with the head's top-N reference images side-by-side.
Empty rows mean the cascade didn't pick anything at that depth.

In [ ]:
def show_cascade_references(selected, fv, layer_names, n_refs=N_REFS):
    rows = [l for l in layer_names if selected.get(l)]
    if not rows:
        print('cascade selected no concepts — nothing to display')
        return None
    max_k = max(len(selected[l]) for l in rows)
    fig, axes = plt.subplots(
        len(rows), max_k * n_refs,
        figsize=(1.2 * max_k * n_refs, 1.4 * len(rows)),
        squeeze=False,
    )
    cascade_concept = fv_results['head'][1]  # HeadConcept(model)
    for r, layer in enumerate(rows):
        ids = list(selected[layer])
        # FV index must cover this layer under the 'head' tag — verified
        # by the existence of <FV_CACHE_DIR>/head/RelMax_*/<layer>_data.npy.
        ref = fv.get_max_reference(
            ids, layer, mode='relevance',
            r_range=(0, n_refs), composite=composite, rf=False,
        )
        for c in range(max_k * n_refs):
            axes[r, c].axis('off')
        for k, cid in enumerate(ids):
            img_list, _heat = ref[cid]
            for i, img in enumerate(img_list[:n_refs]):
                ax = axes[r, k * n_refs + i]
                ax.imshow(img); ax.axis('off')
                if i == 0:
                    # Per-concept-group title: layer (line 1) + id (line 2)
                    # on the first column; just the id on subsequent groups.
                    if k == 0:
                        title = row_label(cascade_concept, cid, layer)
                    else:
                        title = label_id(cascade_concept, cid)
                    ax.set_title(title, fontsize=7, loc='left', family='monospace')
    plt.tight_layout()
    return fig

# The head FV's layer_map covers HEAD_LAYER + every CASCADE_LAYER_NAMES
# entry, so every layer plot_cascade visits has reference samples.
head_fv = fv_results['head'][0]
show_cascade_references(selected, head_fv, CASCADE_LAYER_NAMES)
plt.show()

## 12. Notes & next steps

* **Magnitude regime.** With AlphaBeta(0.5, 0.5) the input |R|_max
  is O(10²) and conservation is within a few × the target logit
  (vs ~10²² magnitudes under the standard `2y+ε` rule — see
  `RESEARCH_NOTES.md` Entry 6).
* **Spatial vs prefix tokens.** All concepts here cleanly separate
  the two: the per-head + AttnOutputDim concepts only see spatial
  patch tokens; `RegisterTokenConcept` only sees the prefix
  (cls + register) tokens. Neither concept class mixes them.
* **`AttnWeightConcept` is intentionally absent.** Softmax weights
  have no fixed semantic per neuron (the same cell combines
  different concepts for different inputs), so reference-sample
  retrieval would be uninformative. The `attn.softmax` submodule
  is still hookable for direct attention-map inspection via
  `record_layer=['backbone.blocks.{i}.attn.softmax']` — useful for K/Q
  relation analysis but not for concept identification.
* **FV indexing scope.** Indexes the whole loaded dataset by
  default (`fv_end = len(dataset)`). For fast iteration during
  development, override `fv_end` in section 9 with a smaller
  integer (e.g. 500). The cache is persisted to
  `data/fv_cache_dinov3_unfolded/<tag>` so subsequent runs reuse it.
* **Other layers.** Re-run sections 5-8 with `HEAD_LAYER` / `Q_LAYER` / … pointed at
  e.g. 0 (early — likely texture concepts) or 23 (late — likely
  object-semantic concepts) to compare across depth.
* **Other concepts in cascade.** Section 11 uses `HeadConcept`;
  swap to `QConcept` / `KConcept` / `VConcept` to compare which
  attention sub-axis dominates the model's reasoning.